# April 2019 event (ow-0009) -- Stage III physical backtracking

**New event, same unmodified methodology.** Uses the same `iste.ensemble_run.
run_ensemble` (T09), same forcing ensemble setup (T08), same explicit 6-member
subset/`master_seed`s/`K` diffusivity/`dt`/`n_steps` as every prior event run in
this project (June 2019, October). The only things that differ are the event's own
real parcels/timestamp (`ow-0009.jpg`) and the new ocean data file
(`data-2/ocean_april2019_aoi_22_to_28.nc`, already on the project's standard
regional grid -- no cropping needed this time).

Outputs saved under `result_ow0009/` (new, working directory for this event's
intermediate artifacts) -- `result/` (June) and `output/ow-0008/` are untouched.

In [1]:
import os, sys, json, time
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import xarray as xr

from iste.observation import adapt_upstream_particles
from iste.environment import load_field_dataset
from iste.forcing_ensemble import ForcingEnsemble, ForcingEnsembleConfig
from iste.ensemble_run import EnsembleRunRecords, run_ensemble
from iste.lookup_storage import ParticleRecordStore

RESULT_DIR = os.path.join(REPO_ROOT, "result_ow0009")
os.makedirs(RESULT_DIR, exist_ok=True)
print(f"ow-0009 outputs will be saved under {RESULT_DIR}")

ow-0009 outputs will be saved under C:\Users\venka\OneDrive\Desktop\SIH-2026\project-v5\result_ow0009


## 1. Load the real observation (`ow-0009.jpg`)

In [2]:
spill_data = json.load(open(os.path.join(REPO_ROOT, "data", "database_payload.json"), encoding="utf-8"))
spill = next(s for s in spill_data["spills"] if s["image_filename"] == "ow-0009.jpg")
assert spill["total_parcels"] == 371 == len(spill["parcels"])

T_OBS = np.datetime64(spill["timestamp"], "ns")
assert T_OBS == np.datetime64("2019-04-28T15:38:41", "ns"), T_OBS
print(f"image_filename : {spill['image_filename']}")
print(f"observation (T0): {T_OBS}")
print(f"total_parcels  : {spill['total_parcels']}")

image_filename : ow-0009.jpg
observation (T0): 2019-04-28T15:38:41.000000000
total_parcels  : 371


## 2. Load the April 2019 `ocean.nc` (already on the project's standard grid -- no crop needed)

In [3]:
OCEAN_PATH = os.path.join(REPO_ROOT, "data-2", "ocean_april2019_aoi_22_to_28.nc")
ocean = load_field_dataset(OCEAN_PATH, ["uo", "vo"], time_dim="time", lat_dim="lat", lon_dim="lon")
print(f"loaded ocean.nc: grid {ocean.lat.shape} x {ocean.lon.shape}, "
      f"time {ocean.time[0]} .. {ocean.time[-1]} ({len(ocean.time)} hourly steps)")

T72 = T_OBS - np.timedelta64(72, "h")
print(f"T0  (t_obs) = {T_OBS}")
print(f"T72         = {T72}")

coverage_ok = (ocean.time[0] <= T72) and (T_OBS <= ocean.time[-1])
print(f"ocean.nc covers the full [T72, T0] window: {coverage_ok}")
assert coverage_ok, "ocean.nc does not cover the required 72h backward window"
margin_before_h = float((T72 - ocean.time[0]) / np.timedelta64(1, 'h'))
margin_after_h = float((ocean.time[-1] - T_OBS) / np.timedelta64(1, 'h'))
print(f"margin before T72: {margin_before_h:.2f}h; margin after T0: {margin_after_h:.2f}h")

assert str(ocean.time[0])[:7] == "2019-04", ocean.time[0]
assert not any(str(t)[:7] not in ("2019-04",) for t in ocean.time), "non-April 2019 timestamps found in ocean.nc!"
print("confirmed: ocean.nc time axis is entirely April 2019")

C:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\argopy\utils\lists.py:38: UserWarning: An error occurred while loading the ERDDAP data fetcher, it will not be available !
<class 'ImportError'>
cannot import name '_quote_string_constraints' from 'erddapy.erddapy' (C:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\erddapy\erddapy.py)
  warnings.warn(
C:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\argopy\utils\lists.py:50: UserWarning: An error occurred while loading the ArgoVis data fetcher, it will not be available !
<class 'ImportError'>
cannot import name '_quote_string_constraints' from 'erddapy.erddapy' (C:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\erddapy\erddapy.py)
  warnings.warn(
C:\Users\venka\AppData\Local\Programs\Python\Python313\Lib\site-packages\argopy\utils\lists.py:66: UserWarning: An error occurred while loading the GDAC data fetcher, it will not be available !
<class 

loaded ocean.nc: grid (149,) x (216,), time 2019-04-22T00:30:00.000000000 .. 2019-04-28T23:30:00.000000000 (168 hourly steps)
T0  (t_obs) = 2019-04-28T15:38:41.000000000
T72         = 2019-04-25T15:38:41.000000000
ocean.nc covers the full [T72, T0] window: True
margin before T72: 87.14h; margin after T0: 7.86h
confirmed: ocean.nc time axis is entirely April 2019


## 3. T07: adapt the 371 real upstream parcels

In [4]:
parcels = spill["parcels"]
particle_records = [
    {
        "particle_id": str(p["id"]), "latitude": p["lat"], "longitude": p["lon"],
        "observation_pixel_id": str(p["id"]), "timestamp": T_OBS,
    }
    for p in parcels
]
adapted = adapt_upstream_particles(particle_records)
assert adapted.n_particles == 371
print(f"T07 adapted {adapted.n_particles} real particles at t_obs={T_OBS} (verified == 371)")

T07 adapted 371 real particles at t_obs=2019-04-28T15:38:41.000000000 (verified == 371)


## 4. Forcing ensemble + K (identical configuration to every prior event run)

In [5]:
FULL_ENSEMBLE_N_MEMBERS_CONFIGURED = 50
FORCING_MEMBER_IDS_USED = (0, 1, 2, 3, 4, 5)
print(f"Configured full ensemble: {FULL_ENSEMBLE_N_MEMBERS_CONFIGURED} members. "
      f"ACTUAL members used: {FORCING_MEMBER_IDS_USED} (same explicit subset as every prior run)")

ensemble_config = ForcingEnsembleConfig(n_members=FULL_ENSEMBLE_N_MEMBERS_CONFIGURED, master_seed=7)
ensemble = ForcingEnsemble(ocean, ensemble_config)
K_DIFFUSIVITY = 3.0  # m^2/s, same constant as every prior run
K = np.full((len(ocean.lat), len(ocean.lon)), K_DIFFUSIVITY)
print(f"K grid shape: {K.shape} (matches ocean grid)")

Configured full ensemble: 50 members. ACTUAL members used: (0, 1, 2, 3, 4, 5) (same explicit subset as every prior run)
K grid shape: (149, 216) (matches ocean grid)


## 5. Stage III: real backward transport for all 371 real particles (April 2019 currents)

Calls `iste.ensemble_run.run_ensemble` (T09, unmodified) -- same `dt=3600s`,
`n_steps=72`, `master_seed=42` as every prior run. No backward SDE is reimplemented.

In [6]:
DT_SECONDS = 3600.0
N_STEPS = 72
EXPECTED_N_RECORDS = 371 * len(FORCING_MEMBER_IDS_USED) * (N_STEPS + 1)
history_path = os.path.join(RESULT_DIR, "stage3_particle_history_ow0009.npz")

print(f"Running Stage III backward transport: {adapted.n_particles} particles x "
      f"{len(FORCING_MEMBER_IDS_USED)} members x {N_STEPS} steps ({N_STEPS * DT_SECONDS / 3600 / 24:.1f} days) ...")
t0 = time.time()
records = run_ensemble(
    adapted, list(FORCING_MEMBER_IDS_USED), N_STEPS, DT_SECONDS,
    ocean, ensemble, ocean.lat, ocean.lon, K, master_seed=42,
)
t1 = time.time()
print(f"done in {t1 - t0:.1f}s ({(t1 - t0) / 60:.1f} min) -> {records.n_records} (particle, member, checkpoint) rows")
assert records.n_records == EXPECTED_N_RECORDS

Running Stage III backward transport: 371 particles x 6 members x 72 steps (3.0 days) ...


done in 712.4s (11.9 min) -> 162498 (particle, member, checkpoint) rows


## 6. Save the complete April 2019 particle history + summary

In [7]:
store = ParticleRecordStore()
store.add_records(records)
print(f"ParticleRecordStore (T11): {len(store)} records, {len(store.checkpoints())} checkpoints")

np.savez_compressed(
    history_path,
    particle_id=records.particle_id, forcing_member_id=records.forcing_member_id,
    observation_pixel_id=records.observation_pixel_id, timestamp=records.timestamp,
    lat=records.lat, lon=records.lon, log_weight=records.log_weight, valid=records.valid,
    death_time=records.death_time, death_lat=records.death_lat,
    death_lon=records.death_lon, death_reason=records.death_reason,
    survival_mark=records.survival_mark,
)
print(f"saved complete April 2019 particle history to {history_path}")

checkpoints = np.unique(records.timestamp)
summary_lines = [
    "Stage III April 2019 (ow-0009) real particle history summary",
    f"n_particles (unique particle_id): {len(set(records.particle_id.tolist()))}",
    f"n_forcing_members: {len(FORCING_MEMBER_IDS_USED)} {FORCING_MEMBER_IDS_USED}",
    f"n_timestamps (checkpoints): {len(checkpoints)}",
    f"n_records total: {records.n_records}",
    f"T0  = {checkpoints.max()}",
    f"T72 = {checkpoints.min()}",
    "",
    "valid rows per checkpoint:",
]
for ts in checkpoints:
    mask = records.timestamp == ts
    n_valid = int(np.sum(records.valid[mask]))
    n_total = int(np.sum(mask))
    summary_lines.append(f"  {np.datetime64(ts,'ns')}: {n_valid}/{n_total} valid")
summary_text = chr(10).join(summary_lines)
print(summary_text)
with open(os.path.join(RESULT_DIR, "stage3_history_summary_ow0009.txt"), "w", encoding="utf-8") as f:
    f.write(summary_text)

ParticleRecordStore (T11): 162498 records, 73 checkpoints


saved complete April 2019 particle history to C:\Users\venka\OneDrive\Desktop\SIH-2026\project-v5\result_ow0009\stage3_particle_history_ow0009.npz
Stage III April 2019 (ow-0009) real particle history summary
n_particles (unique particle_id): 371
n_forcing_members: 6 (0, 1, 2, 3, 4, 5)
n_timestamps (checkpoints): 73
n_records total: 162498
T0  = 2019-04-28T15:38:41.000000000
T72 = 2019-04-25T15:38:41.000000000

valid rows per checkpoint:
  2019-04-25T15:38:41.000000000: 2226/2226 valid
  2019-04-25T16:38:41.000000000: 2226/2226 valid
  2019-04-25T17:38:41.000000000: 2226/2226 valid
  2019-04-25T18:38:41.000000000: 2226/2226 valid
  2019-04-25T19:38:41.000000000: 2226/2226 valid
  2019-04-25T20:38:41.000000000: 2226/2226 valid
  2019-04-25T21:38:41.000000000: 2226/2226 valid
  2019-04-25T22:38:41.000000000: 2226/2226 valid
  2019-04-25T23:38:41.000000000: 2226/2226 valid
  2019-04-26T00:38:41.000000000: 2226/2226 valid
  2019-04-26T01:38:41.000000000: 2226/2226 valid
  2019-04-26T02:38:4

## 7. Validation

In [8]:
checks = []
checks.append(("73 distinct checkpoints (T0..T72)", len(checkpoints) == 73))
checks.append(("T0 == true observation timestamp", checkpoints.max() == T_OBS))
checks.append(("T72 == T0 - 72h", checkpoints.min() == T72))
checks.append(("1-hour spacing throughout", bool(np.all(np.diff(np.sort(checkpoints)) == np.timedelta64(1, 'h')))))
checks.append(("n_records matches expected (371 x 6 x 73)", records.n_records == EXPECTED_N_RECORDS))
checks.append(("DT_SECONDS/N_STEPS/master_seeds unchanged from every prior run", (DT_SECONDS, N_STEPS) == (3600.0, 72)))
for label, ok in checks:
    print(f"[{'OK' if ok else 'FAIL'}] {label}")
assert all(ok for _, ok in checks)
print()
print("Stage III April 2019 (ow-0009) run: ALL VALIDATION CHECKS PASSED")

[OK] 73 distinct checkpoints (T0..T72)
[OK] T0 == true observation timestamp
[OK] T72 == T0 - 72h
[OK] 1-hour spacing throughout
[OK] n_records matches expected (371 x 6 x 73)
[OK] DT_SECONDS/N_STEPS/master_seeds unchanged from every prior run

Stage III April 2019 (ow-0009) run: ALL VALIDATION CHECKS PASSED
